# Домашнее задание
Откуда берутся датасеты? Практический проект по сбору данных и работе с текстами

Цель:
В этом домашнем задании вам предстоит обойти все ловушки серверов, пробраться сквозь страницы HTML-код, собрать себе свой собственный датасет и натренировать на нём модель.


Описание/Пошаговая инструкция выполнения домашнего задания:

# Часть 1. Парсинг.

По аналогии с занятием, возьмите интересующий вас сайт, на котором можно пособирать какие-то данные (и при этом API не предоставляется).


Идеальный датасет должен иметь текстовое описание некоторого объекта и некоторую целевую переменную, соответствующую этому объекту. Например:

Сайт новостей: текстовое описание - сама новость, целевая переменная - количество просмотров новости (можно поделить на число дней с момента даты публикации, чтобы получить “среднее число просмотров в день”).
Сайт с товарами/книгами/фильмами: текстовое описание товара/книги/фильма + средний рейтинг в качестве целевой переменной.
Блоги - тексты заметок + число просмотров.
И любые другие ваши идеи, которые подходят под такой формат.

Напишите свой парсер, который будет бегать по страничкам и автоматически что-то собирать.


Не забывайте, что парсинг - это ответственное мероприятие, поэтому не бомбардируйте несчастные сайты слишком частыми запросами (можно ограничить число запросов в секунду при помощи time.sleep(0.3), вставленного в теле цикла)






In [1]:
import warnings
warnings.filterwarnings("ignore")
import requests      
import numpy as np  
import pandas as pd  
import time          

from fake_useragent import UserAgent
from bs4 import BeautifulSoup
from tqdm import tqdm
from time import sleep

Загружаем текстовое содержание и рейтинг постов с сайта Risk.ru 

In [ ]:
first_page_num = 200000
last_page_num = 221467

df = pd.DataFrame(columns=['URL', 'Title', 'Rating', 'Text'])

errors_num = 0

for number in tqdm(range(first_page_num, last_page_num)):
    page_link = 'https://risk.ru/blog/' + str(number)
    response = requests.get(page_link, headers={'User-Agent': UserAgent().chrome})
    sleep(0.1)
    if response.ok:
        html = response.content
        soup = BeautifulSoup(html, 'html.parser')
        title = str(soup.find('h1', class_ = "topic_head").string)
        text = str(soup.find('p', class_ = "topic-txt").get_text())
        rating = int(soup.find(lambda tag: tag.name == 'div' and tag.get('class') == ['number', 'orangeNumber']).string)
        df.loc[len(df)] = [page_link, title, rating, text]
    else:
        errors_num = errors_num + 1

print('Количество ошибок: ', errors_num)

df.to_csv('risk18k.csv')

 25%|██▌       | 5423/21467 [2:03:19<6:32:02,  1.47s/it] 

# Часть 2. NLP.

Разбейте собранные данные на train/test, отложив 20-30% наблюдений для тестирования.

Примените tf-idf преобразование для текстового описания. Используйте как отдельные токены, так и биграммы, отсейте стоп-слова, а также слова, которые встречаются слишком редко или слишком часто (параметры min/max_df), не забудьте убрать l2 регуляризацию, которая по умолчанию включена.

Если в вашем датасете целевая переменная непрерывная (например, среднее число просмотров в день), то воспользуйтесь линейной регрессией, если дискретная (положительный/отрицательный отзыв), то логистической.

Постройте регрессию с настройкой параметра регуляризации, оцените качество при помощи соответствующих задаче метрик.

Визуализируйте получившиеся коэффициенты регрессии (возьмите топ-50 слов)

Проинтерпретируйте результаты.

P.S. Если с парсингом не задалось или данных собралось слишком мало - не отчаивайтесь, главное, что ваш парсер собрал хоть что-то! А для второй части задания можно скачать данные по отзывам на фильмы с сайта IMDB (https://www.kaggle.com/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews), в которых для каждого отзыва поставлена семантическая оценка - "позитивный" или "негативный".


Критерии оценки:

Получилось написать работающий парсер - 4 балла.

Предобработка текста при помощи tf-idf - 2 балла.

Построение моделей и визуализация результатов - 4 балла.

In [ ]:
df = pd.read_csv('risk18k.csv')

In [ ]:
df_new = df[df['Text'].str.len() > 200]

In [ ]:
df_new.count()

Unnamed: 0    16203
URL           16203
Title         16203
Rating        16203
Text          16203
dtype: int64

In [ ]:
X = df_new['Text']
Y = df_new['Rating']

In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
train_texts, test_texts, train_y, test_y = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
import nltk

In [ ]:
# загружаем список стоп-слов для русского
nltk.download('stopwords')
stop_words = nltk.corpus.stopwords.words('russian')

stop_words = stop_words + ['каждый', 'которые', 'который', 'нам', 'наш', 'также', 'это', 'ru', '10', '30', 'поэтому']


[nltk_data] Downloading package stopwords to /home/petr/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from nltk.tokenize import word_tokenize
import string

nltk.download('punkt')
punctuation = set(string.punctuation)

[nltk_data] Downloading package punkt to /home/petr/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
import spacy

# Загрузка языковой модели spaCy
nlp = spacy.load("ru_core_news_sm")

def preprocess_and_lemmatize(text):
    doc = nlp(text.lower())  # Привести к нижнему регистру и лемматизировать
    lemmatized_words = [token.lemma_ for token in doc if token.text not in punctuation and token.text not in stop_words]
    return " ".join(lemmatized_words)

/home/petr/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
train_texts.describe

<bound method NDFrame.describe of 4815     \n\r\nХейзел Финдлей и Алекс Хоннольд в марте ...
1828     4 мая в Баку завершился 2-й этап Кубка мира-20...
7678     19 июня в Граце завершился четвертый этап моло...
6978     \n\r\nДирекция альпинистской спортивной базы "...
11290    \n\r\nПервый акклиматизационный выход в район ...
                               ...                        
15285    Таймыр . Плато Путорана .\r\nВчера\r\nПисьмо м...
6193     Это песня о вечных неисправимых путешественник...
1071     \r\nКаякинг, безусловно, незимний вид спорта. ...
18018    Пока продолжается регистрация на Забег на Казб...
8299     УВАЖАЕМЫЕ коллеги и друзья!\n\r\nВ октябре 201...
Name: Text, Length: 12962, dtype: object>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Создание объекта TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words=stop_words, 
                                   norm=None, 
                                   ngram_range=(1, 2), 
                                   max_features=50, 
                                   preprocessor=preprocess_and_lemmatize, 
                                   min_df=0.2,
                                   max_df=0.8)

vectorizer.fit(train_texts)

# Получение списка ключевых слов и их значения TF-IDF для первого документа
feature_names = vectorizer.get_feature_names_out()

print(feature_names)

['альпинизм' 'альпинист' 'больший' 'большой' 'вершина' 'весь' 'вид'
 'восхождение' 'время' 'второй' 'высота' 'год' 'гора' 'горный' 'группа'
 'день' 'жизнь' 'знать' 'идти' 'команда' 'маршрут' 'место' 'метр' 'мир'
 'мочь' 'несколько' 'новый' 'нужный' 'очень' 'первый' 'погода' 'пройти'
 'просто' 'путь' 'работа' 'район' 'россия' 'свой' 'сделать' 'сказать'
 'сложный' 'соревнование' 'стать' 'сторона' 'участник' 'фото' 'хороший'
 'час' 'часть' 'человек']


In [ ]:
# Обучаем TF-IDF на train, а затем применяем test
train_X = vectorizer.fit_transform(train_texts)
test_X  = vectorizer.transform(test_texts)

In [ ]:
# Пример
train_X.todense()[:2] # посмотрим на первые 2 строки

matrix([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  1.90807035,  0.        ,  2.27922274,
          3.5567437 ,  2.50222769,  0.        ,  0.        ,  0.        ,
          1.98828841,  0.        ,  0.        ,  0.        ,  0.        ,
          2.2342287 ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          7.77623427,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          4.49406459,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  2.39801127,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  1.77033968,  8.94965706,
          0.        ,  6.15109595,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  2

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error

In [ ]:
lasso = Lasso(alpha=0.1)
lasso.fit(train_X, train_y.to_numpy())
rmse = root_mean_squared_error(test_y.to_numpy(), lasso.predict(test_X))
print('RMSE = ', rmse)

RMSE =  47.19513497024687


In [ ]:
lasso.coef_

array([ 0.42269823,  0.        ,  0.        , -0.39165472,  0.37771097,
        0.51622479, -0.14857589, -0.17869154, -0.18202184,  0.7670429 ,
       -0.22859594,  0.29589376,  0.12730566, -0.63181446, -0.18588859,
        0.02176403,  1.16266309,  1.30620533, -0.03852271,  0.29801104,
        0.48576389, -0.37739114, -0.00600815, -0.22190009,  0.3568489 ,
        0.33075574, -0.4251411 ,  0.        , -0.35214755,  0.        ,
        0.48360894, -0.10623815, -0.01628437, -0.10400588,  0.04271713,
        0.21256642, -0.13061981,  0.10872478,  0.06172195, -0.12630014,
        0.0832541 , -0.47170793,  0.33285949,  0.30688625, -0.29483477,
        0.32313645,  0.        , -0.32449086,  0.21461314,  0.04945985])

In [ ]:
lasso.intercept_

np.float64(42.85613937839418)

In [ ]:
for coef, word in sorted(zip(lasso.coef_, feature_names), reverse=True):
    print(f"%15s  -  {coef}" % word, )

          знать  -  1.3062053259269082
          жизнь  -  1.1626630910498612
         второй  -  0.7670429002645341
           весь  -  0.5162247860216422
        маршрут  -  0.4857638900983017
         погода  -  0.4836089420549845
      альпинизм  -  0.4226982299565326
        вершина  -  0.3777109662141697
           мочь  -  0.35684889798179825
          стать  -  0.33285948638519347
      несколько  -  0.33075574013546016
           фото  -  0.3231364542222425
        сторона  -  0.3068862458525972
        команда  -  0.2980110428726811
            год  -  0.2958937601933139
          часть  -  0.2146131357595925
          район  -  0.2125664197850371
           гора  -  0.12730566461034215
           свой  -  0.10872477832204167
        сложный  -  0.08325410405869686
        сделать  -  0.061721951688431295
        человек  -  0.049459849833719566
         работа  -  0.042717125513017776
           день  -  0.02176403037894319
        хороший  -  0.0
         первый  -  0.0
   

## Интерпретация результатов

Высокий рейтинг имеют посты связанные с безопасностью в альпинизме.

Низкий рейтинг имеют посты о соревнованиях. 